In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'Scripts'))  # helper modules (run_config, obs, standardplots) live in LPNE/Scripts

from run_config import PROJECT  # project root (LPNE/), found relative to run_config.py - no hard-coded paths
import LPNE
import AREmulator
import run_config as rc
import statschecker as sc
import standardplots as sp

import numpy as np
import numpy.random as npr
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
import tensorflow as tf
from tensorflow.keras import layers, Model
import tensorflow.keras.backend as K
import keras as ke
from tensorflow.keras import regularizers
import optuna as opt
seed = 42

### Setup

In [ ]:
# config
DATASET = rc.DATASET        # '1461ka'
INSOL = 'insol_ber90'         # key of rc.STARTS: 'early', 'mid', 'late' or 'default' (80:20 split)
N_FEATURES = rc.NFEATURES

PARAMS = {'conv filters': 30, 'conv kernel': 10, 'dense unit': 28, 
          'dropout rate': 0.3353763108415839, 'learning rate': 0.00038902232764492545, 
          'dwt gamma': 0.6506584806454648, 'dwt beta': 0.4637125670415251, 'dwt eps': 4.1664542731895874e-06, 
          'dwt levels': 2} # taken from optimize_model.ipynb, the 'mid' start optimization set
                           # using the 'mid' start params for comparability (and bc the choice doesn't matter much between sets)

# one config per start time record: the choice is part of the config (and so of the run tag)
cfgs = {start: rc.make_config(DATASET, seed=seed, insol=INSOL, train_start=start, window_kyr = 100, params = PARAMS)
        for start in rc.STARTS}

print("run tag :", cfgs['mid']['run_tag'])
#print("weights :", cfgs['mid']['weights'])
print("params  :", cfgs['mid']['params'])
print("train/val start time: ", cfgs['mid']['training_start'])
print("train/val frac: ", cfgs['mid']['train_frac'])

In [ ]:
# load in and split data
raw_ber90 = rc.load_data(cfgs['mid']) # raw data is the same for each (no arbitrary cutoff)
splits_early = rc.split_data(raw_ber90, cfgs['early'])
splits_mid = rc.split_data(raw_ber90, cfgs['mid'])
splits_late = rc.split_data(raw_ber90, cfgs['late'])

print(f'using an early pleistocene start: {rc.describe_split(splits_early)}')
print(f'using an mid pleistocene start: {rc.describe_split(splits_mid)}')
print(f'using an late pleistocene start: {rc.describe_split(splits_late)}')

# time splits differ between start times, since the test/train boundary moves
etestt, etraint, evalt = splits_early['test_time'], splits_early['train_time'], splits_early['validation_time']
mtestt, mtraint, mvalt = splits_mid['test_time'], splits_mid['train_time'], splits_mid['validation_time']
ltestt, ltraint, lvalt = splits_late['test_time'], splits_late['train_time'], splits_late['validation_time']



In [ ]:
# normalize data (each start gets its own scaler, fit on its own training split)
scale_e, train_e, val_e, test_e = rc.fit_and_transform_splits(splits_early)
scale_m, train_m, val_m, test_m = rc.fit_and_transform_splits(splits_mid)
scale_l, train_l, val_l, test_l = rc.fit_and_transform_splits(splits_late)
# can also get and transform standard deviations of records

# chunk normalized data into windows
X_train_e, Y_train_e, Insol_train_e = LPNE.create_windows(train_e, cfgs['early']['window_size'], cfgs['early']['rollout_length'])
X_val_e, Y_val_e, Insol_val_e = LPNE.create_windows(val_e, cfgs['early']['window_size'], cfgs['early']['rollout_length_validation'])
X_train_m, Y_train_m, Insol_train_m = LPNE.create_windows(train_m, cfgs['mid']['window_size'], cfgs['mid']['rollout_length'])
X_val_m, Y_val_m, Insol_val_m = LPNE.create_windows(val_m, cfgs['mid']['window_size'], cfgs['mid']['rollout_length_validation'])
X_train_l, Y_train_l, Insol_train_l = LPNE.create_windows(train_l, cfgs['late']['window_size'], cfgs['late']['rollout_length'])
X_val_l, Y_val_l, Insol_val_l = LPNE.create_windows(val_l, cfgs['late']['window_size'], cfgs['late']['rollout_length_validation'])

### Training

##### early pleistocene start

In [ ]:
params_e = cfgs['early']['params']
paths_e = rc.run_paths(cfgs['early']['run_tag'])
print("run tag :", cfgs['early']['run_tag'])
print("weights :", paths_e['weights'])

LPNE.set_seeds(cfgs['early']['seed'])
brain_e = LPNE.CNN_model(cfgs['early']['window_size'], params=params_e, n_features=N_FEATURES)

emu_e = AREmulator.RolloutEmulator(brain_e,
                                rollout_length=cfgs['early']['rollout_length'],
                                vali_rollout_length=cfgs['early']['rollout_length_validation'],
                                window_size=cfgs['early']['window_size'],
                                n_features=N_FEATURES,
                                dwt_gamma=params_e['dwt gamma'],
                                dwt_beta=params_e['dwt beta'],
                                dwt_eps=params_e['dwt eps'],
                                dwt_levels=params_e['dwt levels'],
                                noise_level=cfgs['early']['noise_level'],
                                )
emu_e.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=params_e['learning rate'], weight_decay=1e-1, clipnorm=1.0))

dummy_X = tf.zeros((1, cfgs['early']['window_size'], N_FEATURES))
_ = emu_e(dummy_X)

# creates runs/<run_tag>/ and writes config.json with the scaler used for this data
rc.save_run(cfgs['early'], scale_e, splits_early)

check_e = sc.PhysicalAttractorCheckpoint(
    filepath = paths_e['weights'],
    acf_threshold = cfgs['early']['acf_threshold']
)
logger_e = sc.WeightStatsLogger()

history_e = emu_e.fit(
    [X_train_e, Insol_train_e],
    Y_train_e,
    validation_data=([X_val_e, Insol_val_e], Y_val_e),
    epochs=cfgs['early']['epochs'],
    callbacks=[check_e, logger_e],
    batch_size=cfgs['early']['batch_size'],
    validation_batch_size=cfgs['early']['batch_size'],
    shuffle=True,
)

##### mid pleistocene start

In [ ]:
params_m = cfgs['mid']['params']
paths_m = rc.run_paths(cfgs['mid']['run_tag'])
print("run tag :", cfgs['mid']['run_tag'])
print("weights :", paths_m['weights'])

LPNE.set_seeds(cfgs['mid']['seed'])
brain_m = LPNE.CNN_model(cfgs['mid']['window_size'], params=params_m, n_features=N_FEATURES)

emu_m = AREmulator.RolloutEmulator(brain_m,
                                rollout_length=cfgs['mid']['rollout_length'],
                                vali_rollout_length=cfgs['mid']['rollout_length_validation'],
                                window_size=cfgs['mid']['window_size'],
                                n_features=N_FEATURES,
                                dwt_gamma=params_m['dwt gamma'],
                                dwt_beta=params_m['dwt beta'],
                                dwt_eps=params_m['dwt eps'],
                                dwt_levels=params_m['dwt levels'],
                                noise_level=cfgs['mid']['noise_level'],
                                )
emu_m.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=params_m['learning rate'], weight_decay=1e-1, clipnorm=1.0))

dummy_X = tf.zeros((1, cfgs['mid']['window_size'], N_FEATURES))
_ = emu_m(dummy_X)

# creates runs/<run_tag>/ and writes config.json with the scaler used for this data
rc.save_run(cfgs['mid'], scale_m, splits_mid)

check_m = sc.PhysicalAttractorCheckpoint(
    filepath = paths_m['weights'],
    acf_threshold = cfgs['mid']['acf_threshold']
)
logger_m = sc.WeightStatsLogger()

history_m = emu_m.fit(
    [X_train_m, Insol_train_m],
    Y_train_m,
    validation_data=([X_val_m, Insol_val_m], Y_val_m),
    epochs=cfgs['mid']['epochs'],
    callbacks=[check_m, logger_m],
    batch_size=cfgs['mid']['batch_size'],
    validation_batch_size=cfgs['mid']['batch_size'],
    shuffle=True,
)

##### late pleistocene start

In [ ]:
params_l = cfgs['late']['params']
paths_l = rc.run_paths(cfgs['late']['run_tag'])
print("run tag :", cfgs['late']['run_tag'])
print("weights :", paths_l['weights'])

LPNE.set_seeds(cfgs['late']['seed'])
brain_l = LPNE.CNN_model(cfgs['late']['window_size'], params=params_l, n_features=N_FEATURES)

emu_l = AREmulator.RolloutEmulator(brain_l,
                                rollout_length=cfgs['late']['rollout_length'],
                                vali_rollout_length=cfgs['late']['rollout_length_validation'],
                                window_size=cfgs['late']['window_size'],
                                n_features=N_FEATURES,
                                dwt_gamma=params_l['dwt gamma'],
                                dwt_beta=params_l['dwt beta'],
                                dwt_eps=params_l['dwt eps'],
                                dwt_levels=params_l['dwt levels'],
                                noise_level=cfgs['late']['noise_level'],
                                )
emu_l.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=params_l['learning rate'], weight_decay=1e-1, clipnorm=1.0))

dummy_X = tf.zeros((1, cfgs['late']['window_size'], N_FEATURES))
_ = emu_l(dummy_X)

# creates runs/<run_tag>/ and writes config.json with the scaler used for this data
rc.save_run(cfgs['late'], scale_l, splits_late)

check_l = sc.PhysicalAttractorCheckpoint(
    filepath = paths_l['weights'],
    acf_threshold = cfgs['late']['acf_threshold']
)
logger_l = sc.WeightStatsLogger()

history_l = emu_l.fit(
    [X_train_l, Insol_train_l],
    Y_train_l,
    validation_data=([X_val_l, Insol_val_l], Y_val_l),
    epochs=cfgs['late']['epochs'],
    callbacks=[check_l, logger_l],
    batch_size=cfgs['late']['batch_size'],
    validation_batch_size=cfgs['late']['batch_size'],
    shuffle=True,
)

### Testing

In [ ]:
print(f"Loading best physical model for final evaluation from each model:")
print(f"Loading from the early start...")
emu_e.load_weights(str(paths_e['weights']))
print(f"Loading from the mid start...")
emu_m.load_weights(str(paths_m['weights']))
print(f"Loading from the late start...")
emu_l.load_weights(str(paths_l['weights']))

In [ ]:
sp.LossPlots(history_e, check_e)

In [ ]:
sp.LossPlots(history_m, check_m)

In [ ]:
sp.LossPlots(history_l, check_l)

In [ ]:
w = cfgs['early']['window_size']

# rollout seeded from the same oldest window, continued through train and validation to the present.
# Uses the full scaled record instead of joining the splits: `+` on numpy arrays adds them elementwise,
# and split_data leaves a 1-row gap between test and train, so joined splits would skip an age
fulle = scale_e.transform(splits_early['full'])
fullte = splits_early['time']
rollout_full_e = sp.rollout(emu_e, fulle, w)
# boundary between this start's test period and its train/validation period (split_data skips the row in between)
test_end_e = (etestt.min() + etraint.max()) / 2

In [ ]:
w2 = cfgs['mid']['window_size']

# rollout seeded from the same oldest window, continued through train and validation to the present.
# Uses the full scaled record instead of joining the splits: `+` on numpy arrays adds them elementwise,
# and split_data leaves a 1-row gap between test and train, so joined splits would skip an age
fullm = scale_m.transform(splits_mid['full'])
fulltm = splits_mid['time']
rollout_full_m = sp.rollout(emu_m, fullm, w2)
# boundary between this start's test period and its train/validation period (split_data skips the row in between)
test_end_m = (mtestt.min() + mtraint.max()) / 2

In [ ]:
w3 = cfgs['late']['window_size']

# rollout seeded from the same oldest window, continued through train and validation to the present.
# Uses the full scaled record instead of joining the splits: `+` on numpy arrays adds them elementwise,
# and split_data leaves a 1-row gap between test and train, so joined splits would skip an age
fulll = scale_l.transform(splits_late['full'])
fulltl = splits_late['time']
rollout_full_l = sp.rollout(emu_l, fulll, w3)
# boundary between this start's test period and its train/validation period (split_data skips the row in between)
test_end_l = (ltestt.min() + ltraint.max()) / 2

In [ ]:
# all three rollouts on one set of panels. Each run has its own scaler (fit on its own training split), so
# their scaled values aren't comparable as-is: undo each run's scaling, then rescale everything with the mid
# start's scaler. One ground truth (fullm, with its GPR 1-sigma band) then serves all three
from sklearn.metrics import mean_squared_error
import matplotlib.colors as mc

def shade(color, f):
    """f < 0 darkens toward black, f > 0 lightens toward white, by |f| of the way"""
    c = np.array(mc.to_rgb(color))
    return mc.to_hex(c + ((0 if f < 0 else 1) - c) * abs(f))

EMU_COLORS = ["#FFB600", "#6519C5", "#8DB1A7"]   # emulator colors of sp.plot_rollout's palette, for X, Y, Z
SHADES = {'early': -0.5, 'mid': 0.0, 'late': 0.35}  # early darkest -> late lightest

def to_mid_scale(rollout_data, scaler):
    """rollout scaled with `scaler` -> the same rollout scaled with the mid start's scaler"""
    return scale_m.transform(scaler.inverse_transform(rollout_data))

# ground truth on the mid scale, and its GPR 1-sigma (X, Y, Z only; insolation has no variance)
truth = fullm
truth_std = rc.scale_std(splits_mid['full_std'], scale_m)

def plot_start_rollouts(runs, window_size, title):
    """
        runs - start -> rollout on the mid scale. Every rollout is seeded from the oldest window of the record,
               so row k of each one lines up with row k of truth (and fulltm), however long it is
        draws each start's rollout in its shade of the variable's color, the ground truth (black dashed) with
        its 1-sigma band, and each start's train-test break (dotted, same shade)
    """
    leng = max(ro.shape[0] for ro in runs.values())   # truth is drawn over the longest rollout
    t = fulltm[:leng]

    with plt.rc_context({'font.size': 16}):
        fig, axs = plt.subplots(4, 1, figsize=(22, 15), sharex=True)

        for i in range(3):
            axs[i].fill_between(t, truth[:leng, i] - truth_std[:leng, i], truth[:leng, i] + truth_std[:leng, i],
                                color='black', alpha=0.12, lw=0, label=r"real scaled data $\pm \sigma$")
            axs[i].plot(t, truth[:leng, i], color='black', linestyle='dashed', alpha=0.35, lw=2, label="real scaled data")

            for start, ro in runs.items():
                color = shade(EMU_COLORS[i], SHADES[start])
                n = ro.shape[0]
                mse = mean_squared_error(truth[window_size:n, i], ro[window_size:, i])
                axs[i].plot(fulltm[:n], ro[:, i], color=color, lw=2,
                            label=f"{start} start ({cfgs[start]['training_start']} ka), " + r"$\mathcal{L}_{L_2}$" + f" = {mse:.3f}")
                axs[i].axvline(test_ends[start], color=color, linestyle=':', lw=2)

            axs[i].set_ylabel(sp.MAPPINGS[i])
            axs[i].legend(loc='upper left', fontsize=14)
            axs[i].grid(True, alpha=0.3)

        # train-test break of each run, labelled once above the top panel
        for start in runs:
            axs[0].annotate(f"{start}: test | train", xy=(test_ends[start], 1.0), xycoords=('data', 'axes fraction'),
                            xytext=(0, 6), textcoords='offset points', ha='center', va='bottom', fontsize=14)

        # insolation: the same forcing drives all three rollouts
        axs[3].plot(t, truth[:leng, 3], color='coral', lw=1.5)
        for start in runs:
            axs[3].axvline(test_ends[start], color='dimgray', linestyle=':', lw=2)
        axs[3].set_title("Insolation forcing")
        axs[3].set_ylabel("Insolation (scaled)")
        axs[3].grid(True, alpha=0.3)

        axs[0].set_title(title, pad=30)
        axs[-1].invert_xaxis()
        axs[-1].set_xlabel("time [kyr]")
        plt.tight_layout()
        plt.show()

test_ends = {'early': test_end_e, 'mid': test_end_m, 'late': test_end_l}

# full record: seeded from the oldest window, rolled out through test, train and validation to the present
runs_full = {
    'early': to_mid_scale(rollout_full_e, scale_e),
    'mid':   rollout_full_m,
    'late':  to_mid_scale(rollout_full_l, scale_l),
}
plot_start_rollouts(runs_full, w2, "Full-record rollouts by training start (scaled with the mid start's scaler)")

In [ ]:
# test sets only: each start's rollout over its own test period (1461 ka to its training start), so each
# line stops at its own train-test break. Same seed window as the full-record rollouts
runs_test = {
    'early': to_mid_scale(sp.rollout(emu_e, test_e, w), scale_e),
    'mid':   sp.rollout(emu_m, test_m, w2),
    'late':  to_mid_scale(sp.rollout(emu_l, test_l, w3), scale_l),
}
plot_start_rollouts(runs_test, w2, "Test-set rollouts by training start (scaled with the mid start's scaler)")

##### Phase space

In [ ]:
# phase space of each start's full-record rollout, one figure per start (not layered). Uses the rollouts and
# ground truth on the mid scale from above, so the axes mean the same thing across the three figures
for start, ro in runs_full.items():
    print(f"{start} start ({cfgs[start]['training_start']} ka)")
    sp.phase_space(None, truth[:ro.shape[0]], cfgs[start]['window_size'], rollout_data = ro)

##### Frequency spectra (X)

In [ ]:
# multitaper spectra of X (ice mass, dO18) for each start's full-record rollout, one figure per start (not
# layered). Same mid-scale rollouts and ground truth as above; results kept in spectra[start] for later use
spectra = {}
for start, ro in runs_full.items():
    print(f"{start} start ({cfgs[start]['training_start']} ka)")
    spectra[start] = sp.frequency_spectra(None, truth[:ro.shape[0]], cfgs[start]['window_size'],
                                          timestep=cfgs[start]['timestep'],
                                          spec_var=0,             # 0 = X
                                          rollout_data=ro)